In [21]:

import json
import requests
import os
import random
import time
from urllib.parse import urlparse
import shutil
from os import listdir
from os.path import isfile, join

In [22]:
with open('Recommendation_Data/raw_data.json') as f:
    json_dict = json.load(f)

In [23]:
image_links = []
for r in json_dict['review']:
    link_photo = r['image_link']
    image_links.append(link_photo)


In [24]:
def copy_default_image(default_image_path, save_path, verbose=False):
    """
    Copies the default logo image to the destination path.
    
    Args:
        default_image_path (str): Path to the default logo image
        save_path (str): Destination path for the copied image
        verbose (bool): Whether to print status messages
        
    Returns:
        bool: True if successful, False if failed
    """
    try:
        shutil.copy2(default_image_path, save_path)
        if verbose:
            print("Used default logo image instead")
        return True
    except Exception as e:
        if verbose:
            print(f"Error copying default image: {str(e)}")
        return False


def download_image(url, save_folder, default_image_path, verbose=False):
    """
    Downloads an image from a URL and saves it to the specified folder.
    If download fails, uses the default logo image instead.
    
    Args:
        url (str): URL of the image to download
        save_folder (str): Path to the folder where images will be saved
        default_image_path (str): Path to the default logo image
        verbose (bool): Whether to print status messages
    
    Returns:
        bool: True if successful (either download or default image), False if all failed
    """
    try:
        # Create the save folder if it doesn't exist
        if not os.path.exists(save_folder):
            os.makedirs(save_folder)
        
        filename = os.path.basename(urlparse(url).path) + ".jpg"
        save_path = os.path.join(save_folder, filename)
        
        # Check if file already exists
        if os.path.exists(save_path):
            if verbose:
                print(f"Skipping {filename} - already exists")
            return True
            
        # Try to download the image
        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()  # Raise an exception for bad status codes
            
            # Save the image
            with open(save_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            
            if verbose:
                print(f"Successfully downloaded: {filename}")
            return True
        except Exception as download_error:
            if verbose:
                print(f"Error downloading image from {url}: {str(download_error)}")
                print("Using default logo instead...")
            return copy_default_image(default_image_path, save_path, verbose=verbose)
    except Exception as e:
        if verbose:
            print(f"Unexpected error: {str(e)}")
        return False


def batch_download_images(urls, save_folder, default_image_path, verbose=False):
    """
    Downloads multiple images with optional logging and random delays.
    Uses default logo image when downloads fail.
    
    Args:
        urls (list): List of image URLs to download
        save_folder (str): Path to the folder where images will be saved
        default_image_path (str): Path to the default logo image
        verbose (bool): Whether to print status messages
    """
    # Verify default image exists
    if not os.path.exists(default_image_path):
        raise FileNotFoundError(f"Default image not found at: {default_image_path}")
    
    # Keep track of downloads and skips
    total_urls = len(urls)
    downloaded = 0
    skipped = 0
    default_used = 0
    failed = 0
    
    for i, url in enumerate(urls, 1):
        if verbose:
            print(f"Processing image {i} of {total_urls}")
        
        # Check if the file exists before downloading
        filename = os.path.basename(urlparse(url).path) + ".jpg"
        
        save_path = os.path.join(save_folder, filename)
        
        if os.path.exists(save_path):
            skipped += 1
            continue
        
        result = download_image(url, save_folder, default_image_path, verbose=verbose)
        if result:
            if os.path.getsize(save_path) == os.path.getsize(default_image_path):
                default_used += 1
            else:
                downloaded += 1
        else:
            failed += 1
        
        if verbose and i < total_urls:
            delay = random.uniform(1, 2)
            print(f"Waiting {delay:.2f} seconds...")
            time.sleep(delay)
    
    print("\nDownload Summary:")
    print(f"Total URLs processed: {total_urls}")
    print(f"Successfully downloaded: {downloaded}")
    print(f"Default logo used: {default_used}")
    print(f"Skipped (already existed): {skipped}")
    print(f"Failed: {failed}")


In [26]:


# Specify your save folder
save_folder = "Recommendation_images"

# Download the images
batch_download_images(image_links, save_folder, "media/default.png")


Download Summary:
Total URLs processed: 8
Successfully downloaded: 8
Default logo used: 0
Skipped (already existed): 0
Failed: 0


In [27]:
filename = [f for f in listdir(save_folder) if isfile(join(save_folder, f))]

In [28]:
image_dict = {}

for f in filename:
    id = f .replace(".jpg", "")
    new_path_f = save_folder + "/" + f
    for p in image_links:
        if id in p:
            image_dict[p] = new_path_f
            break

len(image_dict) == len(image_links)

True

In [29]:
for i in range(len(json_dict['review'])):
    old_value = json_dict['review'][i]['image_link']
    json_dict['review'][i]['image_link'] = image_dict[old_value]

In [30]:
def decode_unicode_dict(obj):
    if isinstance(obj, dict):
        return {key: decode_unicode_dict(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [decode_unicode_dict(element) for element in obj]
    elif isinstance(obj, str):
        # Using encode/decode to handle Unicode escape sequences
        try:
            return obj.encode('latin-1').decode('utf-8')
        except UnicodeEncodeError:
            # If it's already properly decoded, return as is
            return obj
    else:
        return obj

In [31]:
json_dict_decoded = decode_unicode_dict(json_dict)

In [32]:
with open('Recommendation_Data/cleaned_data.json', 'w', encoding='utf-8') as outfile:
    json.dump(json_dict_decoded, outfile, ensure_ascii=False, indent=2)